# Polynomial Regression and the Bias-Variance Trade-off

## Introduction

**Polynomial regression** extends linear regression by adding polynomial features of the input:

$$\phi(x) = [1,\, x,\, x^2,\, \ldots,\, x^d]^\top$$

The model $\hat{y} = \boldsymbol{\theta}^\top \phi(x)$ is **still linear in the parameters** $\boldsymbol{\theta}$ — we can apply all tools from ordinary linear regression. This is an instance of **basis function regression**.

This notebook uses polynomial regression as a lens to understand one of the deepest concepts in machine learning: the **bias-variance trade-off**. We will see that:
- Low-degree polynomials **underfit** (high bias, low variance)
- High-degree polynomials **overfit** (low bias, high variance)
- **Regularisation** (L1/Lasso, L2/Ridge) can control overfitting
- **Cross-validation** gives an unbiased estimate of generalisation error

### Learning Objectives
1. Understand polynomial feature expansion as basis function regression
2. Visualise the bias-variance trade-off empirically
3. Derive and interpret $L_1$ (Lasso) and $L_2$ (Ridge) regularisation
4. Implement $k$-fold cross-validation to select the optimal model

## Mathematical Background

### Notation

| Symbol | Meaning |
|--------|---------|
| $m$ | Number of training samples |
| $d$ | Polynomial degree |
| $\phi(x) \in \mathbb{R}^{d+1}$ | Polynomial feature vector: $[1, x, \ldots, x^d]^\top$ |
| $\boldsymbol{\Phi} \in \mathbb{R}^{m \times (d+1)}$ | Vandermonde design matrix |
| $\boldsymbol{\theta} \in \mathbb{R}^{d+1}$ | Model parameters |
| $\hat{y} = \boldsymbol{\theta}^\top \phi(x)$ | Model prediction |
| $\mathcal{J}(\boldsymbol{\theta})$ | Regularised cost function |
| $\lambda \geq 0$ | Regularisation strength |

### Polynomial Regression as Basis Function Regression

Given training data $\{(x^{(i)}, y^{(i)})\}_{i=1}^m$, we construct the **Vandermonde matrix**:

$$\boldsymbol{\Phi} = \begin{bmatrix} 1 & x^{(1)} & (x^{(1)})^2 & \cdots & (x^{(1)})^d \\ 1 & x^{(2)} & (x^{(2)})^2 & \cdots & (x^{(2)})^d \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ 1 & x^{(m)} & (x^{(m)})^2 & \cdots & (x^{(m)})^d \end{bmatrix}$$

The OLS solution is $\boldsymbol{\theta}^* = (\boldsymbol{\Phi}^\top \boldsymbol{\Phi})^{-1} \boldsymbol{\Phi}^\top \mathbf{y}$.

### Bias-Variance Decomposition

For any estimator $\hat{f}$ (trained on a particular dataset) of a true function $f$, the expected prediction error on an unseen point $x$ decomposes as:

$$\underbrace{\mathbb{E}\bigl[(y - \hat{f}(x))^2\bigr]}_{\text{Expected MSE}} = \underbrace{\bigl(\mathbb{E}[\hat{f}(x)] - f(x)\bigr)^2}_{\text{Bias}^2} + \underbrace{\text{Var}(\hat{f}(x))}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{Irreducible Noise}}$$

| Regime | Degree | Bias | Variance | Behaviour |
|--------|--------|------|----------|-----------|
| **Underfitting** | Too low | High | Low | Model too simple to capture pattern |
| **Optimal** | Just right | Low | Low | Good generalisation |
| **Overfitting** | Too high | Low | High | Memorises training noise |

> The irreducible noise $\sigma^2$ is the inherent randomness in the data generating process — **no model can reduce it**.

### Regularised Objective

To control overfitting, we add a penalty $\mathcal{R}(\boldsymbol{\theta})$ to the cost:

$$\mathcal{J}(\boldsymbol{\theta}) = \frac{1}{2m}\|\boldsymbol{\Phi}\boldsymbol{\theta} - \mathbf{y}\|^2 + \lambda \mathcal{R}(\boldsymbol{\theta})$$

**L2 (Ridge) regularisation:** $\mathcal{R}(\boldsymbol{\theta}) = \frac{1}{2}\|\boldsymbol{\theta}\|_2^2$
- Closed-form: $\boldsymbol{\theta}^*_{\text{Ridge}} = (\boldsymbol{\Phi}^\top\boldsymbol{\Phi} + \lambda\mathbf{I})^{-1}\boldsymbol{\Phi}^\top\mathbf{y}$
- Effect: shrinks all coefficients smoothly toward zero; always invertible (even if $\boldsymbol{\Phi}^\top\boldsymbol{\Phi}$ is singular)
- Bayesian interpretation: equivalent to a **Gaussian prior** on $\boldsymbol{\theta}$

**L1 (Lasso) regularisation:** $\mathcal{R}(\boldsymbol{\theta}) = \|\boldsymbol{\theta}\|_1 = \sum_j |\theta_j|$
- No closed form; solved via coordinate descent or LARS
- Effect: promotes **sparsity** (exact zeros in $\boldsymbol{\theta}$) → automatic feature selection
- Bayesian interpretation: equivalent to a **Laplace prior** on $\boldsymbol{\theta}$

Geometrically, the Lasso constraint set $\{\boldsymbol{\theta}: \|\boldsymbol{\theta}\|_1 \leq t\}$ is a **diamond** (has corners on axes). The MSE contours tend to touch the diamond at a corner → sparsity. Ridge's spherical constraint has no corners → smooth shrinkage.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline

# Apply a clean, professional plot style globally
plt.style.use('seaborn-v0_8-whitegrid')


Firstly, we will need some polynomial data to start with.

In [ ]:
np.random.seed(42)  # Set the seed for reproducibility

In [ ]:
# Generate training samples
x_train = np.random.rand(100,1)
y_train = - x_train + 3 * (x_train ** 2) - 2 * (x_train ** 3) + 2 + np.random.rand(100,1) * 0.1

# Generate some outlier points in the dataset
x_train_noise = np.random.rand(10,1)
y_train_noise = - x_train_noise + 3 * (x_train_noise ** 2) - 2 * (x_train_noise ** 3) + 2 \
                + np.random.rand(10,1) * 0.5

# Combine 'normal' points and 'outlier' points to a single training set
x_train = np.concatenate((x_train, x_train_noise), axis=0)
y_train = np.concatenate((y_train, y_train_noise), axis=0)

# Generate test samples
x_test = np.random.rand(20,1)
y_test = - x_test + 3 * (x_test ** 2) - 2 * (x_test ** 3) + 2 + np.random.rand(20,1) * 0.1

### Dataset

We use a synthetic 1D dataset generated from a cubic polynomial with noise:

$$y = -x + 3x^2 - 2x^3 + 2 + \varepsilon, \quad \varepsilon \sim \mathcal{U}(0, 1)$$

We have separate **training** and **test** splits, which is essential for evaluating generalisation.

In [ ]:
# Plot training and test samples
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(x_train, y_train, alpha=0.7, label='Training samples')
ax.scatter(x_test, y_test, alpha=0.9, marker='*', s=80, label='Test samples')
ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('y', rotation=0, fontsize=14)
ax.set_title('Synthetic Cubic Polynomial Dataset', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()


## Degree 1: Simple Linear Regression (High Bias / Underfitting)

With $d=1$ we fit $\hat{y} = \theta_0 + \theta_1 x$ — a straight line.

**Expectation:** this will underfit because the true function is cubic. The model has high bias (its functional form cannot represent the truth) but low variance (a line is stable across different datasets).

In [ ]:
# Generate polynomial features (degree=1 → simple linear regression)
polynomial_features = PolynomialFeatures(degree=1)
x_train_poly = polynomial_features.fit_transform(x_train)[:, 1:]
x_test_poly  = polynomial_features.fit_transform(x_test)[:, 1:]

# Fit model
model = LinearRegression()
model.fit(x_train_poly, y_train)

# Report coefficients
print('--- Degree-1 Model Coefficients ---')
coef_str = ', '.join(f'{c:.4f}' for c in model.coef_.ravel())
print(f'  Coefficients : [{coef_str}]')
print(f'  Intercept    : {model.intercept_[0]:.4f}')


In [ ]:
train_mse_d1 = mean_squared_error(model.predict(x_train_poly), y_train)
test_mse_d1  = mean_squared_error(model.predict(x_test_poly),  y_test)
print('--- Degree-1 MSE ---')
print(f'  Train MSE : {train_mse_d1:.4f}')
print(f'  Test  MSE : {test_mse_d1:.4f}')


Let's plot the fitting line we have just trained.

In [ ]:
# Sort training points so the line is drawn without zigzags
idx = np.argsort(x_train, axis=0)[:, 0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_train[idx], model.predict(x_train_poly)[idx], 'r-', lw=2,
        label='Fitted curve (degree 1)')
ax.scatter(x_train, y_train, alpha=0.6, label='Training samples')
ax.scatter(x_test,  y_test,  alpha=0.9, marker='*', s=80, label='Test samples')
ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('y', rotation=0, fontsize=14)
ax.set_title('Degree 1 — Underfitting (High Bias)', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()


## Degree 3: Polynomial Regression (Well-Specified Model)

With $d=3$ we fit $\hat{y} = \theta_0 + \theta_1 x + \theta_2 x^2 + \theta_3 x^3$.

Since the **true data generating process is degree 3**, this model is correctly specified — we expect both train and test error to be low. The model has enough capacity to capture the true structure without memorising noise.

In [ ]:
# Generate polynomial features (degree=3 — well-specified model)
polynomial_features = PolynomialFeatures(degree=3)
x_train_poly = polynomial_features.fit_transform(x_train)[:, 1:]
x_test_poly  = polynomial_features.fit_transform(x_test)[:, 1:]

print(f'Polynomial feature matrix shape: {x_train_poly.shape}')


In [ ]:
# Fit linear regression on cubic features
model = LinearRegression()
model.fit(x_train_poly, y_train)

print('--- Degree-3 Model Coefficients ---')
coef_str = ', '.join(f'{c:.4f}' for c in model.coef_.ravel())
print(f'  Coefficients : [{coef_str}]')
print(f'  Intercept    : {model.intercept_[0]:.4f}')


In [ ]:
train_mse_d3 = mean_squared_error(model.predict(x_train_poly), y_train)
test_mse_d3  = mean_squared_error(model.predict(x_test_poly),  y_test)
print('--- Degree-3 MSE ---')
print(f'  Train MSE : {train_mse_d3:.4f}')
print(f'  Test  MSE : {test_mse_d3:.4f}')


In [ ]:
idx = np.argsort(x_train, axis=0)[:, 0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_train[idx], model.predict(x_train_poly)[idx], 'r-', lw=2,
        label='Fitted curve (degree 3)')
ax.scatter(x_train, y_train, alpha=0.6, label='Training samples')
ax.scatter(x_test,  y_test,  alpha=0.9, marker='*', s=80, label='Test samples')
ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('y', rotation=0, fontsize=14)
ax.set_title('Degree 3 — Well-Specified Model (Good Fit)', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()


## Degree 30: Extreme Overfitting (Low Bias, High Variance)

With $d=30$ we have 31 parameters to fit 100 training points. The model will **memorise** the training data — including the noise — and fail to generalise.

**What to expect:**
- Training MSE ≈ 0 (fits every training point almost exactly)
- Test MSE ≫ training MSE (the "generalisation gap")

In [ ]:
# Generate polynomial features (degree=30 — severe overfitting)
polynomial_features = PolynomialFeatures(degree=30)
x_train_poly = polynomial_features.fit_transform(x_train)[:, 1:]
x_test_poly  = polynomial_features.fit_transform(x_test)[:, 1:]

# Fit model
model = LinearRegression()
model.fit(x_train_poly, y_train)

# Report first few and last few coefficients to avoid overwhelming output
coefs = model.coef_.ravel()
print('--- Degree-30 Model Coefficients (first 5 / last 5) ---')
print(f'  First 5 : {[f"{c:.4f}" for c in coefs[:5]]}')
print(f'  Last  5 : {[f"{c:.4f}" for c in coefs[-5:]]}')
print(f'  Intercept : {model.intercept_[0]:.4f}')


In [ ]:
train_mse_d30 = mean_squared_error(model.predict(x_train_poly), y_train)
test_mse_d30  = mean_squared_error(model.predict(x_test_poly),  y_test)
print('--- Degree-30 MSE ---')
print(f'  Train MSE : {train_mse_d30:.4f}')
print(f'  Test  MSE : {test_mse_d30:.4f}')
print()
print('Note: near-zero train MSE but large test MSE is the hallmark of overfitting.')


As expected, the degree-30 model severely overfits: perfect fit on training data, but wildly oscillating predictions on new data (Runge's phenomenon). The oscillations grow largest near the boundaries of the training data range.

In [ ]:
idx = np.argsort(x_train, axis=0)[:, 0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_train[idx], model.predict(x_train_poly)[idx], 'r-', lw=2,
        label='Fitted curve (degree 30)')
ax.scatter(x_train, y_train, alpha=0.6, label='Training samples')
ax.scatter(x_test,  y_test,  alpha=0.9, marker='*', s=80, label='Test samples')
ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('y', rotation=0, fontsize=14)
ax.set_title('Degree 30 — Severe Overfitting (High Variance)', fontsize=14)
ax.set_ylim(y_train.min() - 0.5, y_train.max() + 0.5)  # clip extreme oscillations
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()


## Bias-Variance Trade-off: Train vs. Test Error vs. Degree

The following plot is the empirical illustration of the bias-variance trade-off. We train polynomial models of increasing degree $d$ and measure both training MSE and test MSE:

- **Training MSE** decreases monotonically with $d$ (more complex model = better fit to training data)
- **Test MSE** initially decreases (bias reduction) then increases (variance increase)
- The optimal $d$ minimises test MSE

> This U-shape in test error is the defining signature of the bias-variance trade-off.

In [ ]:
degree_of_poly = 15
train_scores = []
test_scores  = []

for degree in range(1, degree_of_poly):
    polynomial_features = PolynomialFeatures(degree=degree)
    x_train_poly = polynomial_features.fit_transform(x_train)[:, 1:]
    x_test_poly  = polynomial_features.fit_transform(x_test)[:, 1:]

    model = LinearRegression()
    model.fit(x_train_poly, y_train)

    train_scores.append(mean_squared_error(model.predict(x_train_poly), y_train))
    test_scores.append(mean_squared_error(model.predict(x_test_poly),  y_test))

degrees = list(range(1, degree_of_poly))
best_degree = degrees[int(np.argmin(test_scores))]
best_test_mse = min(test_scores)

# ── Print summary table ──────────────────────────────────
print('Degree | Train MSE | Test MSE')
print('-------+-----------+---------')
for d, tr, te in zip(degrees, train_scores, test_scores):
    marker = ' <-- best' if d == best_degree else ''
    print(f'  {d:>2d}   |  {tr:.4f}   | {te:.4f}{marker}')

# ── Plot ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(degrees, train_scores, 'bo-', lw=2, label='Train MSE')
ax.plot(degrees, test_scores,  'rs-', lw=2, label='Test MSE')

# Annotate the sweet spot (minimum test error)
ax.axvline(best_degree, color='green', linestyle='--', alpha=0.7,
           label=f'Sweet spot (degree={best_degree})')
ax.annotate(
    f'Min test MSE\ndegree={best_degree}\nMSE={best_test_mse:.4f}',
    xy=(best_degree, best_test_mse),
    xytext=(best_degree + 1.2, best_test_mse + 0.003),
    arrowprops=dict(arrowstyle='->', color='green'),
    fontsize=10, color='green'
)

ax.set_xlabel('Polynomial Degree', fontsize=13)
ax.set_ylabel('Mean Squared Error', fontsize=13)
ax.set_title('Bias-Variance Trade-off: Train vs. Test MSE', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


The test error curve has a clear minimum at the optimal degree. Beyond that, the model fits the training noise and **generalises poorly** — a classic sign of overfitting.

### How Does Regularisation Help?

Instead of limiting the polynomial degree, we can keep a high-degree polynomial (rich feature set) and add a regularisation penalty. This allows the model to use its high-degree features *only when they genuinely improve fit*, keeping other coefficients near zero.

In [ ]:
# Generate polynomial features for a selected degree
degree_selected = 10
polynomial_features = PolynomialFeatures(degree=degree_selected)
x_train_poly = polynomial_features.fit_transform(x_train)[:, 1:]
x_test_poly  = polynomial_features.fit_transform(x_test)[:, 1:]

# Fit model
model = LinearRegression()
model.fit(x_train_poly, y_train)

train_mse = mean_squared_error(y_train, model.predict(x_train_poly))
test_mse  = mean_squared_error(y_test,  model.predict(x_test_poly))
print(f'--- Degree-{degree_selected} Model (No Regularisation) ---')
print(f'  Train MSE : {train_mse:.4f}')
print(f'  Test  MSE : {test_mse:.4f}')

# Sort training indices for smooth curve
idx = np.argsort(x_train, axis=0)[:, 0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_train[idx], model.predict(x_train_poly)[idx], 'r-', lw=2,
        label=f'Fitted curve (degree {degree_selected})')
ax.scatter(x_train, y_train, alpha=0.6, label='Training samples')
ax.scatter(x_test,  y_test,  alpha=0.9, marker='*', s=80, label='Test samples')
ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('y', rotation=0, fontsize=14)
ax.set_title(f'Degree {degree_selected} Polynomial Fit (No Regularisation)', fontsize=14)
ax.set_ylim(y_train.min() - 0.3, y_train.max() + 0.3)  # clip extreme oscillations
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
degree_of_poly = 25
train_scores_l1 = []
test_scores_l1  = []

for degree in range(1, degree_of_poly):
    polynomial_features = PolynomialFeatures(degree=degree)
    x_train_poly = polynomial_features.fit_transform(x_train)[:, 1:]
    x_test_poly  = polynomial_features.fit_transform(x_test)[:, 1:]

    model = Lasso(alpha=0.001, max_iter=10000)  # alpha = L1 penalty strength
    model.fit(x_train_poly, y_train)

    train_scores_l1.append(mean_squared_error(model.predict(x_train_poly), y_train))
    test_scores_l1.append(mean_squared_error(model.predict(x_test_poly),  y_test))

degrees_l1 = list(range(1, degree_of_poly))
best_degree_l1  = degrees_l1[int(np.argmin(test_scores_l1))]
best_test_mse_l1 = min(test_scores_l1)

print(f'Best degree under L1 regularisation (alpha=0.001): {best_degree_l1}')
print(f'  Train MSE at best degree : {train_scores_l1[best_degree_l1 - 1]:.4f}')
print(f'  Test  MSE at best degree : {best_test_mse_l1:.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(degrees_l1, train_scores_l1, 'bo-', lw=2, label='Train MSE (L1)')
ax.plot(degrees_l1, test_scores_l1,  'rs-', lw=2, label='Test MSE (L1)')
ax.axvline(best_degree_l1, color='green', linestyle='--', alpha=0.7,
           label=f'Best degree = {best_degree_l1}')
ax.set_xlabel('Polynomial Degree', fontsize=13)
ax.set_ylabel('Mean Squared Error', fontsize=13)
ax.set_title('L1 (Lasso) Regularisation — Train vs. Test MSE by Degree', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


## L1 Regularisation (Lasso)

By adding the Lasso penalty $\lambda\|\boldsymbol{\theta}\|_1$, we force the model to explain the data using only the most important polynomial terms. The Lasso's sparsity-inducing property means many high-degree coefficients will be set exactly to zero.

**Effect of $\lambda$:**
- Small $\lambda$ ≈ no regularisation (overfitting)  
- Large $\lambda$ → very sparse $\boldsymbol{\theta}$ (underfitting)  
- Optimal $\lambda$: cross-validate!

In [ ]:
# Fit a degree-30 Lasso model and inspect coefficient sparsity
polynomial_features = PolynomialFeatures(degree=30)
x_train_poly = polynomial_features.fit_transform(x_train)[:, 1:]
x_test_poly  = polynomial_features.fit_transform(x_test)[:, 1:]

model = Lasso(alpha=0.0001, max_iter=10000)
model.fit(x_train_poly, y_train)

coefs = model.coef_.ravel()
n_features    = len(coefs)
n_nonzero     = int(np.sum(coefs != 0))
n_zero        = n_features - n_nonzero
zeroed_idx    = np.where(coefs == 0)[0]  # 0-based indices of degree powers (1..30)

print('--- Lasso Degree-30 Model (alpha=0.0001) ---')
print(f'  Total features   : {n_features}')
print(f'  Non-zero coefs   : {n_nonzero}')
print(f'  Zeroed-out coefs : {n_zero}')
if n_zero > 0:
    print(f'  Zeroed feature indices (0-based x^1..x^30): {zeroed_idx.tolist()}')
print(f'  Intercept        : {model.intercept_[0]:.4f}')

# Show non-zero coefficients with their power
print()
print('  Power | Coefficient')
print('  ------+------------')
for power, coef in enumerate(coefs, start=1):
    if coef != 0:
        print(f'  x^{power:<3d} | {coef:.4f}')

# ── Coefficient magnitude bar chart ──────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
powers = np.arange(1, n_features + 1)
colors = ['steelblue' if c != 0 else 'lightgray' for c in coefs]
ax.bar(powers, np.abs(coefs), color=colors, edgecolor='k', linewidth=0.4)
ax.set_xlabel('Feature power (x^k)', fontsize=13)
ax.set_ylabel('|Coefficient|', fontsize=13)
ax.set_title('Lasso Coefficient Magnitudes — Degree-30 (alpha=0.0001)', fontsize=14)
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color='steelblue', label='Non-zero'),
    plt.Rectangle((0,0),1,1, color='lightgray',  label='Zeroed out (Lasso sparsity)'),
], fontsize=11)
plt.tight_layout()
plt.show()


## L2 Regularisation (Ridge)

The Ridge penalty $\frac{\lambda}{2}\|\boldsymbol{\theta}\|_2^2$ has a closed-form solution:

$$\boldsymbol{\theta}^*_{\text{Ridge}} = (\boldsymbol{\Phi}^\top\boldsymbol{\Phi} + \lambda\mathbf{I})^{-1}\boldsymbol{\Phi}^\top\mathbf{y}$$

Unlike Lasso, Ridge does **not** produce exact zeros — it smoothly shrinks all coefficients. However, adding $\lambda\mathbf{I}$ to the Gram matrix $\boldsymbol{\Phi}^\top\boldsymbol{\Phi}$ also improves numerical stability (the matrix becomes positive definite for any $\lambda > 0$).

In [ ]:
degree_of_poly = 25
train_scores_l2 = []
test_scores_l2  = []

for degree in range(1, degree_of_poly):
    polynomial_features = PolynomialFeatures(degree=degree)
    x_train_poly = polynomial_features.fit_transform(x_train)[:, 1:]  # drop bias column
    x_test_poly  = polynomial_features.fit_transform(x_test)[:, 1:]

    model = Ridge(alpha=0.001)  # alpha = L2 penalty strength
    model.fit(x_train_poly, y_train)

    train_scores_l2.append(mean_squared_error(y_train, model.predict(x_train_poly)))
    test_scores_l2.append(mean_squared_error(y_test,  model.predict(x_test_poly)))

degrees_l2 = list(range(1, degree_of_poly))
best_degree_l2  = degrees_l2[int(np.argmin(test_scores_l2))]
best_test_mse_l2 = min(test_scores_l2)

print(f'Best degree under L2 regularisation (alpha=0.001): {best_degree_l2}')
print(f'  Train MSE at best degree : {train_scores_l2[best_degree_l2 - 1]:.4f}')
print(f'  Test  MSE at best degree : {best_test_mse_l2:.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(degrees_l2, train_scores_l2, 'bo-', lw=2, label='Train MSE (L2)')
ax.plot(degrees_l2, test_scores_l2,  'rs-', lw=2, label='Test MSE (L2)')
ax.axvline(best_degree_l2, color='green', linestyle='--', alpha=0.7,
           label=f'Best degree = {best_degree_l2}')
ax.set_xlabel('Polynomial Degree', fontsize=13)
ax.set_ylabel('Mean Squared Error', fontsize=13)
ax.set_title('L2 (Ridge) Regularisation — Train vs. Test MSE by Degree', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Fit a degree-30 Ridge model
polynomial_features = PolynomialFeatures(degree=30)
x_train_poly = polynomial_features.fit_transform(x_train)[:, 1:]
x_test_poly  = polynomial_features.fit_transform(x_test)[:, 1:]

model = Ridge(alpha=0.001)
model.fit(x_train_poly, y_train)

train_mse_ridge = mean_squared_error(y_train, model.predict(x_train_poly))
test_mse_ridge  = mean_squared_error(y_test,  model.predict(x_test_poly))
print('--- Ridge Degree-30 Model (alpha=0.001) ---')
print(f'  Train MSE : {train_mse_ridge:.4f}')
print(f'  Test  MSE : {test_mse_ridge:.4f}')
print(f'  Intercept : {model.intercept_[0]:.4f}')

# ── Regularisation path: coefficient magnitudes vs log(alpha) ──
alphas = np.logspace(-4, 2, 60)  # sweep alpha over many orders of magnitude
coef_paths = []

for a in alphas:
    m = Ridge(alpha=a)
    m.fit(x_train_poly, y_train)
    coef_paths.append(m.coef_.ravel())

coef_paths = np.array(coef_paths)  # shape (n_alphas, n_features)

# Find alpha with lowest test MSE
test_mses_path = [
    mean_squared_error(y_test, Ridge(alpha=a).fit(x_train_poly, y_train).predict(x_test_poly))
    for a in alphas
]
optimal_alpha = alphas[int(np.argmin(test_mses_path))]
print(f'\n  Optimal alpha (lowest test MSE on degree-30 features): {optimal_alpha:.4e}')

fig, ax = plt.subplots(figsize=(10, 5))
for k in range(coef_paths.shape[1]):
    ax.plot(alphas, coef_paths[:, k], lw=0.8, alpha=0.5)
ax.axvline(optimal_alpha, color='red', linestyle='--', lw=1.5,
           label=f'Optimal alpha = {optimal_alpha:.4e}')
ax.set_xscale('log')  # alpha varies over orders of magnitude — log scale is essential
ax.set_xlabel('Regularisation strength (alpha)', fontsize=13)
ax.set_ylabel('Coefficient value', fontsize=13)
ax.set_title('Ridge Regularisation Path — Coefficient Magnitude vs. log(alpha)', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


## $k$-Fold Cross-Validation

**Why cross-validation?** Using a single train/test split gives a noisy estimate of generalisation error. $k$-fold CV rotates which fraction is the validation set, giving a more reliable estimate.

**Algorithm:**
1. Partition the training data into $k$ equal folds.
2. For $i = 1, \ldots, k$: train on the other $k-1$ folds, evaluate on fold $i$.
3. Average the $k$ validation scores.

A common choice is $k=5$ or $k=10$. **Leave-One-Out CV (LOOCV)** uses $k=m$ — unbiased but computationally expensive.

> The reported CV score is an estimate of $\mathbb{E}_{\mathcal{D}}[\text{Test Error}]$ (expectation over random training sets), which is much more informative than a single train/test split.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline

degrees = range(1, 16)
cv_means_unreg, cv_stds_unreg = [], []
cv_means_ridge, cv_stds_ridge = [], []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for d in degrees:
    # Without regularisation
    pipe_plain = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('lr',   LinearRegression())
    ])
    scores_plain = cross_val_score(pipe_plain, x_train, y_train.ravel(),
                                   cv=kf, scoring='neg_mean_squared_error')
    cv_means_unreg.append(-scores_plain.mean())
    cv_stds_unreg.append(scores_plain.std())

    # With Ridge regularisation (alpha=0.1)
    pipe_ridge = Pipeline([
        ('poly',  PolynomialFeatures(degree=d, include_bias=False)),
        ('ridge', Ridge(alpha=0.1))
    ])
    scores_ridge = cross_val_score(pipe_ridge, x_train, y_train.ravel(),
                                   cv=kf, scoring='neg_mean_squared_error')
    cv_means_ridge.append(-scores_ridge.mean())
    cv_stds_ridge.append(scores_ridge.std())

cv_means_unreg = np.array(cv_means_unreg)
cv_stds_unreg  = np.array(cv_stds_unreg)
cv_means_ridge = np.array(cv_means_ridge)
cv_stds_ridge  = np.array(cv_stds_ridge)
degrees_list   = list(degrees)

best_unreg = degrees_list[int(np.argmin(cv_means_unreg))]
best_ridge = degrees_list[int(np.argmin(cv_means_ridge))]

print(f'Best degree (no regularisation) : {best_unreg}  '
      f'(CV MSE = {cv_means_unreg[best_unreg - 1]:.4f} ± {cv_stds_unreg[best_unreg - 1]:.4f})')
print(f'Best degree (Ridge, alpha=0.1)  : {best_ridge}  '
      f'(CV MSE = {cv_means_ridge[best_ridge - 1]:.4f} ± {cv_stds_ridge[best_ridge - 1]:.4f})')

fig, ax = plt.subplots(figsize=(10, 5))

# Mean lines
ax.plot(degrees_list, cv_means_unreg, 'bo-', lw=2, label='No regularisation')
ax.plot(degrees_list, cv_means_ridge, 'rs-', lw=2, label=r'Ridge ($\lambda=0.1$)')

# Shaded std bands
ax.fill_between(degrees_list,
                cv_means_unreg - cv_stds_unreg,
                cv_means_unreg + cv_stds_unreg,
                alpha=0.15, color='blue', label='±1 std (no reg.)')
ax.fill_between(degrees_list,
                cv_means_ridge - cv_stds_ridge,
                cv_means_ridge + cv_stds_ridge,
                alpha=0.15, color='red', label=r'±1 std (Ridge)')

# Annotate best values
ax.axvline(best_unreg, color='blue', linestyle=':', alpha=0.6,
           label=f'Best (no reg.) = degree {best_unreg}')
ax.axvline(best_ridge, color='red',  linestyle=':', alpha=0.6,
           label=f'Best (Ridge)   = degree {best_ridge}')

ax.set_xlabel('Polynomial Degree', fontsize=13)
ax.set_ylabel('5-Fold CV MSE', fontsize=13)
ax.set_title('Cross-Validation Error vs. Polynomial Degree\n'
             '(with ±1 std shading)', fontsize=14)
ax.set_yscale('log')  # log scale reveals the wide range of MSE values
ax.legend(fontsize=10, ncol=2)
plt.tight_layout()
plt.show()


## Reflection Questions

1. **Bias-variance decomposition.** For a degree-1 polynomial fitted to data generated by a degree-3 polynomial, which term dominates the expected test error: bias², variance, or irreducible noise? What about a degree-100 polynomial?

2. **Optimal degree.** If the true data-generating process is a degree-$d^*$ polynomial, will OLS polynomial regression of degree $d^*$ have zero bias? What if $d < d^*$? What if $d > d^*$ (but $d$ is finite)?

3. **Lasso sparsity.** Explain geometrically why the $\ell_1$ constraint set $\{\boldsymbol{\theta}: \|\boldsymbol{\theta}\|_1 \leq t\}$ tends to produce sparse solutions, while the $\ell_2$ constraint $\{\boldsymbol{\theta}: \|\boldsymbol{\theta}\|_2^2 \leq t\}$ does not.

4. **Ridge and multicollinearity.** Explain why Ridge regression is preferred over OLS when $\boldsymbol{\Phi}^\top\boldsymbol{\Phi}$ is near-singular. What is the condition number of $(\boldsymbol{\Phi}^\top\boldsymbol{\Phi} + \lambda\mathbf{I})$ compared to $\boldsymbol{\Phi}^\top\boldsymbol{\Phi}$?

5. **Elastic Net.** The Elastic Net combines $L_1$ and $L_2$: $\mathcal{R}(\boldsymbol{\theta}) = \alpha\|\boldsymbol{\theta}\|_1 + \frac{1-\alpha}{2}\|\boldsymbol{\theta}\|_2^2$. When would you choose Elastic Net over pure Lasso or pure Ridge?

## Answers to Reflection Questions

**1. Bias-variance for different degrees.**
For a cubic truth with degree-1 fit: **bias² dominates** — the model family cannot represent the true function regardless of how much data we have. For degree-100: **variance dominates** — each training set gives a wildly different high-degree polynomial; the model memorises noise. Irreducible noise $\sigma^2$ is constant throughout.

---

**2. Optimal degree and bias.**
If the truth is exactly degree $d^*$ and we fit with degree $d^*$: OLS is **unbiased** (the true parameters are in the hypothesis class, so $\mathbb{E}[\hat{f}(x)] = f(x)$). If $d < d^*$: positive bias (model is too simple to represent the truth). If $d > d^*$: still unbiased (the extra coefficients are estimated near zero on average), but variance increases.

---

**3. Lasso sparsity — geometric explanation.**
The OLS solution minimises the MSE loss; its contours are ellipses in $\boldsymbol{\theta}$-space. The constrained solution is where the smallest MSE ellipse first touches the constraint set. The $\ell_1$ ball $\{\|\boldsymbol{\theta}\|_1 \leq t\}$ is a diamond with **corners on the axes**. For any direction of the ellipse axes, the constraint boundary is most likely touched at a corner — where one or more $\theta_j = 0$. The $\ell_2$ ball is a sphere with no corners, so the touching point has all components non-zero.

---

**4. Ridge and multicollinearity.**
If columns of $\boldsymbol{\Phi}$ are nearly linearly dependent, some eigenvalues $\lambda_i$ of $\boldsymbol{\Phi}^\top\boldsymbol{\Phi}$ are near zero, making $\kappa = \lambda_{\max}/\lambda_{\min} \gg 1$ (ill-conditioned). Adding $\lambda\mathbf{I}$: condition number becomes $\frac{\lambda_{\max}+\lambda}{\lambda_{\min}+\lambda}$, which is much closer to 1 for $\lambda \gg \lambda_{\min}$. The Ridge solution is always unique because $(\boldsymbol{\Phi}^\top\boldsymbol{\Phi} + \lambda\mathbf{I})$ has all eigenvalues $\geq \lambda > 0$.

---

**5. Elastic Net.**
The Elastic Net combines $L_1$ and $L_2$: $\mathcal{R} = \alpha\|\boldsymbol{\theta}\|_1 + \frac{1-\alpha}{2}\|\boldsymbol{\theta}\|_2^2$. Prefer Elastic Net when: (a) you have many correlated features (pure Lasso arbitrarily selects one; Elastic Net tends to select groups of correlated features together), (b) $n \gg p$ (many features, fewer samples), (c) you want sparsity but with more stability than Lasso. The Lasso is a special case ($\alpha=1$) and Ridge is the other ($\alpha=0$).